# Project 96 — Local Documentation Writer
## Generate README and Usage Notes from Code and Context

**Stack:** Ollama · LangChain · Jupyter

## Project Overview

This notebook builds a **local documentation writer** that analyzes source code
files and generates comprehensive documentation — README files, API references,
usage guides, and installation notes — all powered by a local LLM via Ollama.

Everything runs **locally**. No code or context leaves your machine.

### What You Will Learn

1. How to extract **code structure** (classes, functions, signatures) for documentation
2. How to generate a complete **README.md** from code context
3. How to generate **API reference** documentation per class
4. How to generate **usage examples** and quick-start guides
5. How to assemble multi-section documentation from separate LLM calls
6. Practical prompt-engineering patterns for documentation generation

## Prerequisites

| Requirement | Details |
|---|---|
| **Ollama** | Running locally with `qwen3:8b` pulled |
| **Python packages** | `langchain`, `langchain-ollama` |

```bash
ollama pull qwen3:8b
```

In [ ]:
# !pip install -q langchain langchain-ollama

## Step 1 — Imports and Configuration

In [ ]:
import re
import textwrap

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

OLLAMA_MODEL = "qwen3:8b"
TEMPERATURE = 0.3

print("Configuration ready.")

## Step 2 — Initialize LLM

We use **Qwen3 8B** via Ollama. A slightly higher temperature (0.3)
helps produce more natural-sounding documentation.

In [ ]:
llm = ChatOllama(model=OLLAMA_MODEL, temperature=TEMPERATURE)

test_msg = llm.invoke("Reply with only: OK")
print(f"LLM ready: {test_msg.content.strip()[:20]}")

## Step 3 — Define Sample Project Source Code

We define a small sample project with two Python files:
- `auth.py` — JWT authentication service
- `cache.py` — LRU cache with TTL support

These represent a realistic micro-library that needs documentation.

In [ ]:
PROJECT_NAME = "pyservices"
PROJECT_DESCRIPTION = "A lightweight Python library for JWT authentication and in-memory caching."

SOURCE_FILES = [
    {
        "name": "auth.py",
        "code": textwrap.dedent('''\
            import jwt
            from datetime import datetime, timedelta
            from hashlib import sha256

            class AuthService:
                """JWT-based authentication service."""

                def __init__(self, secret_key: str, token_expiry: int = 3600):
                    self.secret_key = secret_key
                    self.token_expiry = token_expiry

                def hash_password(self, password: str) -> str:
                    """Hash a password using SHA-256."""
                    return sha256(password.encode()).hexdigest()

                def create_token(self, user_id: int, roles: list[str]) -> str:
                    """Create a JWT token for the given user."""
                    payload = {
                        "sub": user_id,
                        "roles": roles,
                        "exp": datetime.utcnow() + timedelta(seconds=self.token_expiry),
                        "iat": datetime.utcnow(),
                    }
                    return jwt.encode(payload, self.secret_key, algorithm="HS256")

                def verify_token(self, token: str) -> dict | None:
                    """Verify and decode a JWT token. Returns None if invalid."""
                    try:
                        return jwt.decode(token, self.secret_key, algorithms=["HS256"])
                    except jwt.ExpiredSignatureError:
                        return None
                    except jwt.InvalidTokenError:
                        return None

                def refresh_token(self, token: str) -> str | None:
                    """Refresh an existing token if it is still valid."""
                    payload = self.verify_token(token)
                    if payload:
                        return self.create_token(payload["sub"], payload["roles"])
                    return None
        ''')
    },
    {
        "name": "cache.py",
        "code": textwrap.dedent('''\
            from typing import Any, Optional
            from time import time

            class LRUCache:
                """In-memory LRU cache with TTL (time-to-live) support."""

                def __init__(self, max_size: int = 100, ttl: int = 300):
                    self.max_size = max_size
                    self.ttl = ttl
                    self.cache: dict[str, tuple[Any, float]] = {}
                    self.access_order: list[str] = []

                def get(self, key: str) -> Optional[Any]:
                    """Retrieve a value by key. Returns None if missing or expired."""
                    if key in self.cache:
                        value, timestamp = self.cache[key]
                        if time() - timestamp < self.ttl:
                            self.access_order.remove(key)
                            self.access_order.append(key)
                            return value
                        else:
                            del self.cache[key]
                            self.access_order.remove(key)
                    return None

                def set(self, key: str, value: Any) -> None:
                    """Store a key-value pair. Evicts the oldest entry if full."""
                    if key in self.cache:
                        self.access_order.remove(key)
                    elif len(self.cache) >= self.max_size:
                        oldest = self.access_order.pop(0)
                        del self.cache[oldest]
                    self.cache[key] = (value, time())
                    self.access_order.append(key)

                def invalidate(self, key: str) -> bool:
                    """Remove a key from the cache. Returns True if found."""
                    if key in self.cache:
                        del self.cache[key]
                        self.access_order.remove(key)
                        return True
                    return False

                def stats(self) -> dict:
                    """Return cache statistics."""
                    return {"size": len(self.cache), "max_size": self.max_size, "ttl": self.ttl}
        ''')
    },
]

print(f"Project: {PROJECT_NAME}")
print(f"Source files: {len(SOURCE_FILES)}")
for f in SOURCE_FILES:
    lines = f['code'].strip().count('\n') + 1
    print(f"  {f['name']} ({lines} lines)")

## Step 4 — Extract Code Structure

Before sending to the LLM, we extract class names, method signatures,
and docstrings using regex. This gives the LLM structured context
and lets us verify the generated docs cover all methods.

In [ ]:
def extract_code_structure(code: str) -> dict:
    """Extract classes, methods, and docstrings from Python source code."""
    classes = []
    # Find class definitions
    class_pattern = re.compile(r'^class\s+(\w+)(?:\(.*?\))?:', re.MULTILINE)
    for match in class_pattern.finditer(code):
        cls_name = match.group(1)

        # Find methods in this class
        methods = []
        method_pattern = re.compile(r'def\s+(\w+)\(self(?:,\s*(.+?))?\)(?:\s*->\s*(.+?))?:', re.MULTILINE)
        for m in method_pattern.finditer(code[match.start():]):
            methods.append({
                "name": m.group(1),
                "params": m.group(2) or "",
                "return_type": (m.group(3) or "").strip(),
            })

        # Find class docstring
        docstring = ""
        ds_match = re.search(r'class\s+' + cls_name + r'.*?:\s*\n\s+"""(.+?)"""', code, re.DOTALL)
        if ds_match:
            docstring = ds_match.group(1).strip()

        classes.append({"name": cls_name, "docstring": docstring, "methods": methods})

    return {"classes": classes}


for f in SOURCE_FILES:
    structure = extract_code_structure(f["code"])
    f["structure"] = structure
    print(f"\n{f['name']}:")
    for cls in structure["classes"]:
        print(f"  class {cls['name']}: {cls['docstring'][:60]}")
        for m in cls["methods"]:
            ret = f" -> {m['return_type']}" if m['return_type'] else ""
            print(f"    def {m['name']}({m['params'][:40]}){ret}")

## Step 5 — Generate README.md

The README is the most important documentation file. We generate a
complete README with sections: title, description, installation,
quick start, API overview, examples, and license.

In [ ]:
README_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a technical writer. Generate a complete README.md for
the given Python project. The README must include these sections:

# Project Name
- One-paragraph description
- Feature bullet list

## Installation
- pip install command
- Requirements

## Quick Start
- Minimal working example for each class

## API Reference
- Brief description of each class and its key methods

## Examples
- 2-3 realistic usage examples with code blocks

## Contributing
- Brief contribution guidelines

## License
- MIT license note

Return ONLY valid Markdown. Use proper code fences for examples."""),
    ("human", """Project: {project_name}
Description: {description}

Source files:
{code_summary}""")
])

readme_chain = README_PROMPT | llm | StrOutputParser()

# Build code summary for the prompt
code_summary_parts = []
for f in SOURCE_FILES:
    code_summary_parts.append(f"--- {f['name']} ---\n{f['code'].strip()}")
code_summary = "\n\n".join(code_summary_parts)

readme = readme_chain.invoke({
    "project_name": PROJECT_NAME,
    "description": PROJECT_DESCRIPTION,
    "code_summary": code_summary,
})

print("GENERATED README.md")
print("=" * 60)
print(readme[:2000])
if len(readme) > 2000:
    print(f"\n... ({len(readme)} chars total)")

## Step 6 — Generate API Reference per Class

We generate detailed API documentation for each class, covering
constructor parameters, all methods, return types, and examples.

In [ ]:
API_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Generate detailed API reference documentation in Markdown for
the given Python class. For each method include:

- Method signature
- Description (what it does)
- Parameters with types and descriptions
- Return value description
- Exceptions raised (if any)
- A short usage example

Return ONLY valid Markdown."""),
    ("human", "Module: {module}\n\n```python\n{code}\n```")
])

api_chain = API_PROMPT | llm | StrOutputParser()

api_docs = {}
for f in SOURCE_FILES:
    doc = api_chain.invoke({
        "module": f["name"],
        "code": f["code"].strip(),
    })
    api_docs[f["name"]] = doc
    print(f"\nAPI REFERENCE — {f['name']}")
    print("=" * 50)
    print(doc[:1000])
    if len(doc) > 1000:
        print(f"... ({len(doc)} chars total)")
    print()

## Step 7 — Generate Usage Guide

Beyond API reference, developers need a **usage guide** that shows
how to use the classes together in realistic scenarios.

In [ ]:
USAGE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Write a practical usage guide for the given Python project.
Show how to use the classes in realistic scenarios:

1. A web API authentication flow using AuthService
2. Caching API responses using LRUCache
3. Combining both: cached authenticated endpoints

Include complete, runnable code examples with comments.
Explain each step. Return ONLY valid Markdown."""),
    ("human", "Project: {project_name}\n\nSource code:\n{code_summary}")
])

usage_chain = USAGE_PROMPT | llm | StrOutputParser()

usage_guide = usage_chain.invoke({
    "project_name": PROJECT_NAME,
    "code_summary": code_summary,
})

print("USAGE GUIDE")
print("=" * 60)
print(usage_guide[:2000])
if len(usage_guide) > 2000:
    print(f"\n... ({len(usage_guide)} chars total)")

## Step 8 — Generate CHANGELOG

We generate a sample CHANGELOG entry for the project, simulating
what a v1.0.0 release would look like.

In [ ]:
CHANGELOG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Write a CHANGELOG.md entry for the v1.0.0 release of this project.
Follow the Keep a Changelog format:

## [1.0.0] - 2026-04-15
### Added
- feature descriptions
### Security
- security notes

Base the content on the actual classes and methods in the code.
Return ONLY valid Markdown."""),
    ("human", "Project: {project_name}\nModules: {modules}")
])

changelog_chain = CHANGELOG_PROMPT | llm | StrOutputParser()

modules_desc = ", ".join(
    f"{f['name']} ({f['structure']['classes'][0]['name']})"
    for f in SOURCE_FILES if f["structure"]["classes"]
)

changelog = changelog_chain.invoke({
    "project_name": PROJECT_NAME,
    "modules": modules_desc,
})

print("CHANGELOG")
print("=" * 50)
print(changelog)

## Step 9 — Documentation Completeness Check

We verify that the generated README covers all classes and methods
found in the source code.

In [ ]:
print("DOCUMENTATION COMPLETENESS CHECK")
print("=" * 60)

readme_lower = readme.lower()

for f in SOURCE_FILES:
    print(f"\n{f['name']}:")
    for cls in f["structure"]["classes"]:
        found = cls["name"].lower() in readme_lower
        status = "✓" if found else "✗ MISSING"
        print(f"  {status} class {cls['name']}")

        for m in cls["methods"]:
            if m["name"] == "__init__":
                continue
            found = m["name"].lower() in readme_lower
            status = "✓" if found else "✗"
            print(f"    {status} {m['name']}()")

# Check README sections
expected_sections = ["installation", "quick start", "api", "example", "license"]
print(f"\nREADME sections:")
for section in expected_sections:
    found = section in readme_lower
    status = "✓" if found else "✗ MISSING"
    print(f"  {status} {section}")

## Step 10 — Full Documentation Summary

We summarize all generated documentation artifacts.

In [ ]:
print("DOCUMENTATION GENERATION SUMMARY")
print("=" * 60)
print()

artifacts = [
    ("README.md", len(readme)),
    ("API Reference — auth.py", len(api_docs.get("auth.py", ""))),
    ("API Reference — cache.py", len(api_docs.get("cache.py", ""))),
    ("Usage Guide", len(usage_guide)),
    ("CHANGELOG", len(changelog)),
]

total_chars = 0
print(f"{'Artifact':<30} {'Size':>8}")
print("-" * 40)
for name, size in artifacts:
    total_chars += size
    print(f"{name:<30} {size:>6} chars")
print("-" * 40)
print(f"{'TOTAL':<30} {total_chars:>6} chars")
print(f"\nSource code: {sum(len(f['code']) for f in SOURCE_FILES)} chars")
print(f"Doc-to-code ratio: {total_chars / sum(len(f['code']) for f in SOURCE_FILES):.1f}x")

## Evaluation Summary

| Dimension | How We Evaluated |
|---|---|
| **README completeness** | Checked all classes and methods are mentioned (Step 9) |
| **Section coverage** | Verified README has install, quick start, API, examples, license |
| **API reference quality** | Inspected per-class docs for method coverage and examples |
| **Usage guide quality** | Checked for realistic, runnable code examples |
| **Doc-to-code ratio** | Measured total documentation vs. source code size |
| **Code structure extraction** | Verified `extract_code_structure()` finds all classes/methods |

### Known Limitations

- **No runtime validation**: Generated code examples are not executed,
  so they may contain subtle errors.
- **Single-file context**: Each API doc is generated per-file. Cross-file
  references (e.g., using AuthService with LRUCache) rely on the usage guide.
- **Style consistency**: Different LLM calls may produce slightly different
  formatting styles within the same doc set.
- **No docstring extraction**: The structure extractor uses regex, not AST,
  so complex patterns (decorators, nested classes) may be missed.
- **Markdown only**: The generator produces Markdown. For Sphinx/MkDocs,
  additional formatting would be needed.

## How to Improve This Project

1. **AST-based extraction** — use Python's `ast` module for accurate
   class/function/decorator parsing
2. **File-based I/O** — read real `.py` files and write `.md` files to disk
3. **Template system** — use Jinja2 templates for consistent doc formatting
4. **Docstring inheritance** — if classes have existing docstrings, use them
   as seeds rather than generating from scratch
5. **Multi-format output** — generate Sphinx RST, MkDocs YAML, or
   OpenAPI specs alongside Markdown
6. **Incremental updates** — detect code changes and regenerate only
   affected documentation sections

## What You Learned

- **Code structure extraction** — parsing classes and methods for documentation context
- **README generation** — producing a complete README.md from code context
- **API reference generation** — per-class documentation with examples
- **Usage guide generation** — realistic multi-class workflow examples
- **Completeness verification** — checking generated docs cover all code symbols
- **Prompt engineering** — crafting prompts for consistent, well-structured documentation